#**Hybrid Retrieval-Augmented Generation (RAG) with Multi-Query and Reranking**
##**Knowledge Base Creation**
Create a chroma db with both **SparseVectorIndexing** and **DenseVectorIndexing** with the data present in the data folder

###**0. Install Packages**

In [1]:
# ! pip install langchain

In [2]:
# ! pip install langchain-community

In [3]:
# ! pip install langchain-openai

In [4]:
# ! pip install langchain-chroma

In [5]:
# ! pip install snowballstemmer

###**1. Document Loading**

In [6]:
# Load the documents

from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader('./data/subtitles', glob='*.srt', show_progress=True, loader_cls=TextLoader)

documents = loader.load()

print(f'Loaded {len(documents)} documents')

100%|██████████| 10/10 [00:00<00:00, 55.53it/s]

Loaded 10 documents


###**2. Chunking**

In [7]:
# Chunk the loaded documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(f'Created {len(chunks)} chunks')


Created 514 chunks


###**3. Embedding**

In [8]:
import chromadb
from chromadb import Schema, SparseVectorIndexConfig, VectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction, OpenAIEmbeddingFunction
import uuid

In [9]:
# Set up the API keys

# Embedding key
keyFile = open('keys/.azure_openai_embedding_key.txt', 'r')
AZURE_OPENAI_EMBEDDING_API_KEY = keyFile.read().strip()
keyFile.close()

# Chroma key
keyFile = open("keys/.chroma_api_key.txt")
CHROMA_API_KEY = keyFile.read().strip()
keyFile.close()


In [10]:
# Embedding function

embedding_function = OpenAIEmbeddingFunction(
    api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
    api_base="https://gopi-m5ncld8s-eastus2.openai.azure.com/",
    model_name="text-embedding-3-small",
    api_version="2024-12-01-preview"
)

In [11]:
# Create Schema config for Chroma db
schema = Schema()

In [12]:
# Spare vector indexing schema config
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

In [13]:
# Vector indexing schema 

schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
        embedding_function=embedding_function
    )
)

In [14]:
# Create collection
client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant="096092a7-6461-4b3a-bbc6-3a9811f7aaa1",
  database='rag_training_db'
)

collection = client.get_or_create_collection(
  name="friends_collection",
  schema=schema,
)

In [15]:
# Free tier in chroma supports 300 docs to be inserted at a time

BATCH_SIZE = 250

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]

    collection.add(
        documents=[chunk.page_content for chunk in batch],
        metadatas=[chunk.metadata for chunk in batch],
        ids=[str(uuid.uuid4()) for _ in batch]
    )

    print(f"Inserted {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)}")

Inserted 250/514
Inserted 500/514
Inserted 514/514
